# Qwen3-0.6B Tutorial: Naive Transformers Loop vs nano-vllm

This notebook follows the flow from `TUTORIAL.md` and the prompt formatting from `example.py`.
It compares two ways to run **Qwen3-0.6B**:

1. A **naive Transformers implementation** that does autoregressive decoding in Python, one request at a time
2. **nano-vllm**, the vLLM-style inference engine implemented in this repository

The goal is to make the differences visible, measurable, and easy to study.

---
## Part 0: Setup

In [ ]:
import gc
import os
import sys
import time

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

sys.path.insert(0, os.path.dirname(os.path.abspath(".")))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

print(f"PyTorch {torch.__version__}")
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA: {torch.version.cuda}")

In [ ]:
# Point this to your local Qwen3-0.6B weights.
MODEL_PATH = os.path.expanduser("~/huggingface/Qwen3-0.6B/")
if not os.path.isdir(MODEL_PATH):
    MODEL_PATH = "Qwen/Qwen3-0.6B"

TEMPERATURE = 0.6
MAX_NEW_TOKENS = 128
PROMPT_TEXTS = [
    "introduce yourself",
    "list all prime numbers within 100",
]

print(f"Model: {MODEL_PATH}")

We use the same chat-template formatting as `example.py` so the comparison stays aligned with the repo's main example.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

chat_prompts = [
    tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
    for prompt in PROMPT_TEXTS
]

prompt_token_counts = [len(tokenizer(prompt).input_ids) for prompt in chat_prompts]
total_prompt_tokens = sum(prompt_token_counts)

for i, prompt in enumerate(chat_prompts, start=1):
    print(f"Prompt {i} ({prompt_token_counts[i-1]} tokens):")
    print(prompt[:300])
    print()

---
## Part 1: Naive Baseline

This baseline uses HuggingFace Transformers directly and performs decoding in an explicit Python loop.
It is intentionally simple:

- One request at a time
- One token sampled per Python iteration
- No scheduler
- No paged KV cache
- No continuous batching

That makes it a useful reference implementation for studying what nano-vllm improves.

In [ ]:
hf_model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, torch_dtype=DTYPE)
hf_model = hf_model.to(DEVICE)
hf_model.eval()

print(f"HF model dtype: {next(hf_model.parameters()).dtype}")
if DEVICE == "cuda":
    print(f"GPU memory after load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
def sample_next_token(logits, temperature):
    logits = logits / temperature
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)


@torch.inference_mode()
def naive_generate_one(model, tokenizer, prompt, temperature=0.6, max_new_tokens=128):
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs["input_ids"].to(DEVICE)
    attention_mask = inputs["attention_mask"].to(DEVICE)

    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=True,
    )
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    prefill_time = time.perf_counter() - t0

    next_token = sample_next_token(outputs.logits[:, -1, :], temperature)
    past_key_values = outputs.past_key_values
    generated_ids = [next_token.item()]
    decode_time = 0.0

    for _ in range(max_new_tokens - 1):
        if generated_ids[-1] == tokenizer.eos_token_id:
            break

        attention_mask = torch.cat(
            [attention_mask, attention_mask.new_ones((attention_mask.shape[0], 1))],
            dim=1,
        )

        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        outputs = model(
            input_ids=next_token,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            use_cache=True,
        )
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        decode_time += time.perf_counter() - t0

        past_key_values = outputs.past_key_values
        next_token = sample_next_token(outputs.logits[:, -1, :], temperature)
        generated_ids.append(next_token.item())

    return {
        "text": tokenizer.decode(generated_ids, skip_special_tokens=True),
        "token_ids": generated_ids,
        "num_prompt_tokens": int(input_ids.shape[1]),
        "num_generated_tokens": len(generated_ids),
        "prefill_time_s": prefill_time,
        "decode_time_s": decode_time,
        "total_time_s": prefill_time + decode_time,
    }

In [ ]:
def benchmark_naive(model, tokenizer, prompts, temperature=0.6, max_new_tokens=128, warmup=1, repeats=3):
    for _ in range(warmup):
        for prompt in prompts:
            naive_generate_one(model, tokenizer, prompt, temperature=temperature, max_new_tokens=8)

    prefill_times = []
    decode_times = []
    total_times = []
    generated_tokens = []
    last_outputs = None

    for _ in range(repeats):
        run_outputs = []
        prefill_time = 0.0
        decode_time = 0.0
        total_time = 0.0
        num_generated = 0

        if DEVICE == "cuda":
            torch.cuda.reset_peak_memory_stats()

        for prompt in prompts:
            result = naive_generate_one(
                model,
                tokenizer,
                prompt,
                temperature=temperature,
                max_new_tokens=max_new_tokens,
            )
            run_outputs.append(result)
            prefill_time += result["prefill_time_s"]
            decode_time += result["decode_time_s"]
            total_time += result["total_time_s"]
            num_generated += result["num_generated_tokens"]

        prefill_times.append(prefill_time)
        decode_times.append(decode_time)
        total_times.append(total_time)
        generated_tokens.append(num_generated)
        last_outputs = run_outputs

    prefill_time = float(np.median(prefill_times))
    decode_time = float(np.median(decode_times))
    total_time = float(np.median(total_times))
    num_generated = int(np.median(generated_tokens))

    return {
        "prefill_time_ms": prefill_time * 1000,
        "decode_time_ms": decode_time * 1000,
        "total_time_ms": total_time * 1000,
        "num_prompt_tokens": total_prompt_tokens,
        "num_generated_tokens": num_generated,
        "prefill_tok_per_s": total_prompt_tokens / prefill_time,
        "decode_tok_per_s": num_generated / decode_time if decode_time > 0 else 0.0,
        "outputs": last_outputs,
    }

In [ ]:
naive_results = benchmark_naive(
    hf_model,
    tokenizer,
    chat_prompts,
    temperature=TEMPERATURE,
    max_new_tokens=MAX_NEW_TOKENS,
)

print("=" * 60)
print("Naive Transformers Results")
print("=" * 60)
print(f"Prefill  ({naive_results['num_prompt_tokens']} prompt tokens total):")
print(f"  Time:       {naive_results['prefill_time_ms']:.1f} ms")
print(f"  Throughput: {naive_results['prefill_tok_per_s']:.0f} tok/s")
print(f"\nDecode   ({naive_results['num_generated_tokens']} generated tokens total):")
print(f"  Time:       {naive_results['decode_time_ms']:.1f} ms")
print(f"  Throughput: {naive_results['decode_tok_per_s']:.1f} tok/s")
print(f"\nTotal:     {naive_results['total_time_ms']:.1f} ms")
if DEVICE == "cuda":
    print(f"GPU peak memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

for i, result in enumerate(naive_results["outputs"], start=1):
    print("\n" + "-" * 60)
    print(f"Prompt {i}: {PROMPT_TEXTS[i-1]!r}")
    print(result["text"])


### What makes this baseline naive?

It is not wrong. It is just missing the systems optimizations described in `TUTORIAL.md`:

- Every decode step re-enters Python
- Requests are processed sequentially instead of continuously batched
- KV cache growth is tied to each request directly instead of managed in paged blocks
- There is no scheduler separating prefill from decode across many active sequences

That is exactly the gap nano-vllm is designed to close.

In [ ]:
naive_results_saved = {k: v for k, v in naive_results.items() if k != "outputs"}
naive_peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9 if DEVICE == "cuda" else 0.0

del hf_model
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

print("HF baseline cleaned up.")

---
## Part 2: nano-vllm

Now we run the same prompts through the engine built in this repo.
This follows `example.py` directly:

- `LLM(MODEL_PATH, enforce_eager=True, tensor_parallel_size=1)`
- `SamplingParams(temperature=0.6, max_tokens=...)`
- `tokenizer.apply_chat_template(...)` for chat-style prompts

In [ ]:
from nanovllm import LLM, SamplingParams

llm = LLM(MODEL_PATH, enforce_eager=True, tensor_parallel_size=1)
sampling_params = SamplingParams(temperature=TEMPERATURE, max_tokens=MAX_NEW_TOKENS)

print("nano-vllm engine loaded.")
if DEVICE == "cuda":
    print(f"GPU memory after load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

### Engine flow to keep in mind

This maps the notebook back to the source files discussed in `TUTORIAL.md`:

1. `LLM.generate(...)` adds each request to the scheduler
2. The scheduler chooses a prefill or decode batch
3. `ModelRunner` prepares tensors and KV-cache slot mappings
4. The model runs and samples the next token
5. Finished sequences are returned while unfinished ones stay in the engine

That architecture is why nano-vllm can batch work more efficiently than the naive loop.

In [ ]:
nano_outputs = llm.generate(chat_prompts, sampling_params, use_tqdm=False)

for i, output in enumerate(nano_outputs, start=1):
    print("\n" + "-" * 60)
    print(f"Prompt {i}: {PROMPT_TEXTS[i-1]!r}")
    print(output["text"])


In [ ]:
def benchmark_nanovllm(llm, prompts, sampling_params, warmup=1, repeats=3):
    one_token_params = SamplingParams(
        temperature=sampling_params.temperature,
        max_tokens=1,
        ignore_eos=sampling_params.ignore_eos,
    )

    for _ in range(warmup):
        llm.generate(prompts, one_token_params, use_tqdm=False)

    prefill_times = []
    total_times = []
    generated_tokens = []
    last_outputs = None

    for _ in range(repeats):
        if DEVICE == "cuda":
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        llm.generate(prompts, one_token_params, use_tqdm=False)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        prefill_times.append(time.perf_counter() - t0)

        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        outputs = llm.generate(prompts, sampling_params, use_tqdm=False)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        total_times.append(time.perf_counter() - t0)
        generated_tokens.append(sum(len(output["token_ids"]) for output in outputs))
        last_outputs = outputs

    prefill_time = float(np.median(prefill_times))
    total_time = float(np.median(total_times))
    num_generated = int(np.median(generated_tokens))
    decode_time = total_time - prefill_time

    return {
        "prefill_time_ms": prefill_time * 1000,
        "decode_time_ms": decode_time * 1000,
        "total_time_ms": total_time * 1000,
        "num_prompt_tokens": total_prompt_tokens,
        "num_generated_tokens": num_generated,
        "prefill_tok_per_s": total_prompt_tokens / prefill_time,
        "decode_tok_per_s": num_generated / decode_time if decode_time > 0 else 0.0,
        "outputs": last_outputs,
    }

In [ ]:
nano_results = benchmark_nanovllm(llm, chat_prompts, sampling_params)

print("=" * 60)
print("nano-vllm Results")
print("=" * 60)
print(f"Prefill  ({nano_results['num_prompt_tokens']} prompt tokens total):")
print(f"  Time:       {nano_results['prefill_time_ms']:.1f} ms")
print(f"  Throughput: {nano_results['prefill_tok_per_s']:.0f} tok/s")
print(f"\nDecode   ({nano_results['num_generated_tokens']} generated tokens total):")
print(f"  Time:       {nano_results['decode_time_ms']:.1f} ms")
print(f"  Throughput: {nano_results['decode_tok_per_s']:.1f} tok/s")
print(f"\nTotal:     {nano_results['total_time_ms']:.1f} ms")
if DEVICE == "cuda":
    print(f"GPU peak memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

---
## Part 3: Comparison

In [ ]:
nano_peak_memory_gb = torch.cuda.max_memory_allocated() / 1e9 if DEVICE == "cuda" else 0.0

print(f"{'Metric':<30} {'Naive':>14} {'nano-vllm':>14}")
print("-" * 62)
rows = [
    ("Prefill time (ms)", naive_results_saved["prefill_time_ms"], nano_results["prefill_time_ms"]),
    ("Prefill throughput (tok/s)", naive_results_saved["prefill_tok_per_s"], nano_results["prefill_tok_per_s"]),
    ("Decode time (ms)", naive_results_saved["decode_time_ms"], nano_results["decode_time_ms"]),
    ("Decode throughput (tok/s)", naive_results_saved["decode_tok_per_s"], nano_results["decode_tok_per_s"]),
    ("Total time (ms)", naive_results_saved["total_time_ms"], nano_results["total_time_ms"]),
    ("Generated tokens", naive_results_saved["num_generated_tokens"], nano_results["num_generated_tokens"]),
]

for name, naive_value, nano_value in rows:
    print(f"{name:<30} {naive_value:>14.2f} {nano_value:>14.2f}")

if DEVICE == "cuda":
    print("-" * 62)
    print(f"{'Peak memory (GB)':<30} {naive_peak_memory_gb:>14.2f} {nano_peak_memory_gb:>14.2f}")

## What to study next

After running the notebook, read these files side by side with the results:

- `nanovllm/engine/llm_engine.py` for the `generate()` loop
- `nanovllm/engine/scheduler.py` for prefill vs decode scheduling
- `nanovllm/engine/model_runner.py` for tensor preparation and KV-cache handling
- `nanovllm/engine/block_manager.py` for paged KV allocation
- `nanovllm/models/qwen3.py` for the actual Qwen3 architecture

A good exercise is to change the number of prompts, prompt lengths, or `max_tokens` and watch how the gap changes.

In [ ]:
# Optional cleanup
llm.exit()